In [ ]:
%pip install pandas>=2.2 networkx decorator tqdm six scipy
%pip install littleballoffur --no-deps

In [ ]:
pip install torch_geometric

In [ ]:
pip install networkit

## Greedy Graph Coreset Selection Algorithm

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from littleballoffur import RandomWalkSampler, RandomNodeSampler, DegreeBasedSampler, PageRankBasedSampler, ForestFireSampler, MetropolisHastingsRandomWalkSampler, SnowBallSampler
import random
import warnings

warnings.filterwarnings("ignore")


def OMP_sampling(L,NIt):

    n, _ = np.shape(L)
    b = np.zeros((n,1))
    x = []

    for i in range(NIt):

        IND = np.zeros((n,1))
        idx_list = random.sample(range(n), k=100)
        xa = abs(L[:,idx_list].T.dot(b))
        mx = min(xa)

        IND = (xa<=mx)

        Ind = [index for index,value in enumerate(IND) if value]
        Ind = Ind[random.randrange(len(Ind))]
        x.append(idx_list[Ind])

        b = b - L[:,[idx_list[Ind]]]
        L[:,[idx_list[Ind]]] = np.inf*np.ones((n,1))

    return x

## Testing Function

In [ ]:
from scipy.sparse.linalg import eigsh


def build_signal(f_type, n, K, L):
    """
    Build a test signal f : V -> R.

    Parameters
    ----------
    f_type  : 'smooth' | 'linear' | 'degree' | 'random'
    n       : number of nodes
    cluster : (n,) array of cluster assignments
    K       : number of clusters
    L       : (n, n) graph Laplacian (used for degree-based signal)

    Returns
    -------
    f : (n,) ndarray
    """
    if f_type == 'smooth':
        # cluster-constant: low frequency, lives in span{u_0,...,u_{K-1}}
        vals = np.arange(1, K + 1)
        f    = vals[cluster].astype(float)

    elif f_type == 'pw_random':
        # random linear combination of first K eigenvectors
        _, U = np.linalg.eigh(L.todense())
        U_K  = U[:, :K]
        #alpha = np.sqrt(n)*np.random.standard_normal(K)
        alpha = np.random.random(K)*np.sqrt(n)
        f    = U_K @ alpha

    elif f_type == 'pw_uniform':
        # uniform combination: f = sum of first K eigenvectors
        _, U = np.linalg.eigh(L.todense())
        f    = U[:, :K].sum(axis=1)

    elif f_type == 'pw_weighted':
        # frequency-weighted: emphasises lower modes
        eigvals, U = np.linalg.eigh(L.todense())
        eigvals_K  = eigvals[:K]
        alpha      = 1.0 / (eigvals_K + 1e-3)
        f          = U[:, :K] @ alpha

    elif f_type == 'linear':
        # ramps across node index: mid frequency
        f = np.arange(n, dtype=float)

    elif f_type == 'degree':
        # correlated with graph structure
        f = np.array(L.diagonal(), dtype=float)

    elif f_type == 'random':
        # pure noise: high frequency, no cluster structure
        rng = np.random.default_rng(0)
        f   = rng.standard_normal(n)

    else:
        raise ValueError(f"unknown f_type '{f_type}'")

    return f


def single_function_test(f, sample_indices, label='sample'):
    """
    Test whether sample S gives a good estimate of mean(f).

    Parameters
    ----------
    f              : (n,) signal defined on all nodes
    sample_indices : list or array of sampled node indices
    label          : name for this sampling method

    Returns
    -------
    dict with mu_true, mu_sample, error, rel_error
    """
    mu_true   = f.mean()
    mu_sample = f[sample_indices].mean()
    error     = abs(mu_sample - mu_true)
    rel_error = error / (abs(mu_true) + 1e-10)

    return {
        'mu_true'  : mu_true,
        'mu_sample': mu_sample,
        'error'    : error,
        'rel_error': rel_error,
    }

## Cluster Balance Error

In [ ]:
n1, n2, n3 = 2000, 1000, 500
sizes = [n1, n2, n3]
cluster = np.concatenate([0*np.ones(n1), 1*np.ones(n2), 2*np.ones(n3)])
probs = [[0.01, 0.001, 0.001], [0.001, 0.02, 0.001], [0.001, 0.001, 0.04]]

iter = 10
num_sampled = 100

cluster_balance_omp_avg = 0
cluster_balance_RN_avg = 0
cluster_balance_PR_avg = 0
cluster_balance_DB_avg = 0
cluster_balance_MH_avg = 0
cluster_balance_uni_avg = 0

for i in range(iter):

    print(i)
    graph = nx.stochastic_block_model(sizes, probs)

    A = nx.adjacency_matrix(graph)
    n, _ = np.shape(A)
    AA = nx.adjacency_matrix(graph)
    L = nx.laplacian_matrix(graph)
    LL = nx.laplacian_matrix(graph)
    omp_graph = OMP_sampling(L,num_sampled)

    model = RandomNodeSampler(number_of_nodes=num_sampled,seed=None)
    RN_graph = model.sample(graph)

    model = PageRankBasedSampler(number_of_nodes=num_sampled,seed=None)
    PR_graph = model.sample(graph)

    model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
    DB_graph = model.sample(graph)

    model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
    MH_graph = model.sample(graph)

    uniform_graph = list(np.random.choice(list(graph.nodes()), size=num_sampled, replace=False))

    r_uniform = [0]*len(sizes)
    r_RN = [0]*len(sizes)
    r_PR = [0]*len(sizes)
    r_DB = [0]*len(sizes)
    r_MH = [0]*len(sizes)
    r_omp = [0]*len(sizes)

    for i in range(len(sizes)):
        if i == 0:
            r_uniform[i] = len(list(np.nonzero((np.asarray(uniform_graph) < sizes[i]))[0]))/sizes[i]
        else:
            r_uniform[i] = len(list(np.nonzero((np.asarray(uniform_graph) >= sizes[i-1]) & (np.asarray(uniform_graph) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    for i in range(len(sizes)):
        if i == 0:
            r_RN[i] = len(list(np.nonzero((np.asarray(RN_graph.nodes) < sizes[i]))[0]))/sizes[i]
        else:
            r_RN[i] = len(list(np.nonzero((np.asarray(RN_graph.nodes) >= sizes[i-1]) & (np.asarray(RN_graph.nodes) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    for i in range(len(sizes)):
        if i == 0:
            r_PR[i] = len(list(np.nonzero((np.asarray(PR_graph.nodes) < sizes[i]))[0]))/sizes[i]
        else:
            r_PR[i] = len(list(np.nonzero((np.asarray(PR_graph.nodes) >= sizes[i-1]) & (np.asarray(PR_graph.nodes) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    for i in range(len(sizes)):
        if i == 0:
            r_DB[i] = len(list(np.nonzero((np.asarray(DB_graph.nodes) < sizes[i]))[0]))/sizes[i]
        else:
            r_DB[i] = len(list(np.nonzero((np.asarray(DB_graph.nodes) >= sizes[i-1]) & (np.asarray(DB_graph.nodes) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    for i in range(len(sizes)):
        if i == 0:
            r_MH[i] = len(list(np.nonzero((np.asarray(MH_graph.nodes) < sizes[i]))[0]))/sizes[i]
        else:
            r_MH[i] = len(list(np.nonzero((np.asarray(MH_graph.nodes) >= sizes[i-1]) & (np.asarray(MH_graph.nodes) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    for i in range(len(sizes)):
        if i == 0:
            r_omp[i] = len(list(np.nonzero((np.asarray(omp_graph) < sizes[i]))[0]))/sizes[i]
        else:
            r_omp[i] = len(list(np.nonzero((np.asarray(omp_graph) >= sizes[i-1]) & (np.asarray(omp_graph) < sizes[i-1]+sizes[i]))[0]))/sizes[i]

    cluster_balance_omp = 0
    cluster_balance_RN = 0
    cluster_balance_PR = 0
    cluster_balance_DB = 0
    cluster_balance_MH = 0
    cluster_balance_uni = 0

    for i in range(len(r_RN)):
        cluster_balance_omp = cluster_balance_omp + abs(r_omp[i] - len(omp_graph)/sum(sizes))/len(sizes)
        cluster_balance_RN = cluster_balance_RN + abs(r_RN[i] - len(RN_graph)/sum(sizes))/len(sizes)
        cluster_balance_PR = cluster_balance_PR + abs(r_PR[i] - len(PR_graph)/sum(sizes))/len(sizes)
        cluster_balance_DB = cluster_balance_DB + abs(r_DB[i] - len(DB_graph)/sum(sizes))/len(sizes)
        cluster_balance_MH = cluster_balance_MH + abs(r_MH[i] - len(MH_graph)/sum(sizes))/len(sizes)
        cluster_balance_uni = cluster_balance_uni + abs(r_uniform[i] - len(uniform_graph)/sum(sizes))/len(sizes)

    cluster_balance_omp_avg = cluster_balance_omp_avg + cluster_balance_omp/iter
    cluster_balance_RN_avg = cluster_balance_RN_avg + cluster_balance_RN/iter
    cluster_balance_PR_avg = cluster_balance_PR_avg + cluster_balance_PR/iter
    cluster_balance_DB_avg = cluster_balance_DB_avg + cluster_balance_DB/iter
    cluster_balance_MH_avg = cluster_balance_MH_avg + cluster_balance_MH/iter
    cluster_balance_uni_avg = cluster_balance_uni_avg + cluster_balance_uni/iter


print(cluster_balance_omp_avg)
print(cluster_balance_RN_avg)
print(cluster_balance_PR_avg)
print(cluster_balance_DB_avg)
print(cluster_balance_MH_avg)
print(cluster_balance_uni_avg)

## Mean Estimation for Band-limited Function

In [ ]:
n1, n2, n3 = 2000, 1000, 500
sizes = [n1, n2, n3]
cluster = np.concatenate([0*np.ones(n1), 1*np.ones(n2), 2*np.ones(n3)])
probs = [[0.01, 0.001, 0.001], [0.001, 0.02, 0.001], [0.001, 0.001, 0.04]]

iter = 10
num_sampled = 100


mu_true = 0
results_omp_average_error, results_omp_average_rel_error = 0, 0
results_RN_average_error, results_RN_average_rel_error = 0, 0
results_PR_average_error, results_PR_average_rel_error = 0, 0
results_DB_average_error, results_DB_average_rel_error = 0, 0
results_MH_average_error, results_MH_average_rel_error = 0, 0
results_uniform_average_error, results_uniform_average_rel_error = 0, 0

for i in range(iter):

    print(i)
    graph = nx.stochastic_block_model(sizes, probs)

    A = nx.adjacency_matrix(graph)
    n, _ = np.shape(A)
    AA = nx.adjacency_matrix(graph)
    L = nx.laplacian_matrix(graph)
    LL = nx.laplacian_matrix(graph)
    omp_graph = OMP_sampling(L,num_sampled)

    model = RandomNodeSampler(number_of_nodes=num_sampled,seed=None)
    RN_graph = model.sample(graph)

    model = PageRankBasedSampler(number_of_nodes=num_sampled,seed=None)
    PR_graph = model.sample(graph)

    model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
    DB_graph = model.sample(graph)

    model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
    MH_graph = model.sample(graph)

    uniform_graph = set(np.random.choice(list(graph.nodes()), size=num_sampled, replace=False))

    f_type = 'pw_random'
    f = build_signal(f_type, n, 3, LL)
    ## print(f"signal type: {f_type},  true mean μ = {f.mean():.3f}")

    mu_true = mu_true + abs(f.mean()/iter)

    results_omp = single_function_test(f, omp_graph, label='OMP')
    results_RN = single_function_test(f, RN_graph, label='Random Node')
    results_PR = single_function_test(f, PR_graph, label='Page Rank')
    results_DB = single_function_test(f, DB_graph, label='Degree Based')
    results_MH = single_function_test(f, MH_graph, label='MetropolisHastings')
    results_uniform = single_function_test(f, list(uniform_graph), label='uniform')

    results_omp_average_error = results_omp_average_error + results_omp['error']/iter
    results_omp_average_rel_error = results_omp_average_rel_error + results_omp['rel_error']/iter
    results_RN_average_error = results_RN_average_error + results_RN['error']/iter
    results_RN_average_rel_error = results_RN_average_rel_error + results_RN['rel_error']/iter
    results_PR_average_error = results_PR_average_error + results_PR['error']/iter
    results_PR_average_rel_error = results_PR_average_rel_error + results_PR['rel_error']/iter
    results_DB_average_error = results_DB_average_error + results_DB['error']/iter
    results_DB_average_rel_error = results_DB_average_rel_error + results_DB['rel_error']/iter
    results_MH_average_error = results_MH_average_error + results_MH['error']/iter
    results_MH_average_rel_error = results_MH_average_rel_error + results_MH['rel_error']/iter
    results_uniform_average_error = results_uniform_average_error + results_uniform['error']/iter
    results_uniform_average_rel_error = results_uniform_average_rel_error + results_uniform['rel_error']/iter

print(mu_true)

print(results_omp_average_error)
print(results_RN_average_error)
print(results_PR_average_error)
print(results_DB_average_error)
print(results_MH_average_error)
print(results_uniform_average_error)